<a href="https://colab.research.google.com/github/elenamaggiore/Thesis_Elena_Maggiore/blob/preprocessing/PREPROCESSING_masking.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PREPROCESSING: masking

This notebook will address the dataset's problems and prepare them for its use in the analysis.

* First, a preprocessing through REGEX will be applied.

## GENERAL:

Libraries, mounting Google Drive, and reading files.

In [ ]:
from __future__ import annotations
import sys
import subprocess
import os
import gc
import re
import html
import ast
import hashlib
import itertools
import json
import math
import random
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from google.colab import drive
drive.mount('/content/drive')

try:
    import torch
except ImportError:
    torch = None

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## MASKING FILTER

This section will recognise the following elements from users' comments:
* Mention of MBTI types, case insensitive, considering both singular and plural.
* Mention of Cognitive Functions.
* Specific words and expressions like "personality test", "myers - briggs", the Big Five categories, and so on.
* URLs.
* Comments that are generated by bots.
* Comments that are deleted.

In [ ]:
CHUNK_SIZE = 20_000

MIN_COMMENT_WORDS = 5
MAX_COMMENT_WORDS = 500
MASK_LEAKY_COMMENTS = True

TYPE_RE = re.compile(
    r"\b[EI][NS][TF][JP](?:s|'s)?\b", re.I
)

FUNC_RE = re.compile(
    r"(?<![A-Za-z0-9])(?<!Wi-)[NSTF][ie](?:[-/ ]?[NSTF][ie])*(?![A-Za-z0-9])"
)

PLACEHOLDER_RE = re.compile(
    r"\b[A-Z]+\_MENTION\b"
)

VOCAB_RE = re.compile(
    r"\b(?:mbti|myers[- ]?briggs|enneagram|socionics|typology|cognitive functions?|16 ?personalit(?:y|ies)|"
    r"personality (?:test|type|quiz)s?|big ?(?:five|5)|neo[- ]?(?:pi|ffi)|ipip|truity|hexaco|"
    r"openness|conscientiousness|extraversion|extroversion|agreeableness|neuroticism)\b",
    re.I
)

LEAK_PATTERNS = (
    TYPE_RE,
    FUNC_RE,
    PLACEHOLDER_RE,
    VOCAB_RE
)

REPLACEMENTS = [
    (TYPE_RE, "TYPE_MENTION"),
    (FUNC_RE, "COGFUNC_MENTION"),
]

URL_RE = re.compile(
    r"https?://\S+|www\.\S+",
    re.I
)

BOT_RE = re.compile(
    r"i am a bot|this action was performed automatically|bleep,? bloop",
    re.I
)

DELETED = {"[deleted]", "[removed]"}

These functions will be used to determine if a specific comment is a source of data leak, labelling it _ok_, _leaky_, _short_, _with placeholder_, or _bad_. Then, bad and short comments will be removed, leaky comments will be fixed with placeholders and kept together with the rest.

In [ ]:
def author_hash(a): # Creates HASH for author pseudonymisation
  return hashlib.sha256(str(a).encode()).hexdigest()[:10]


def clean_comment(raw): # Clears comments
    if not isinstance(raw, str):
        return None

    t = html.unescape(raw).strip()

    if t.lower() in DELETED or BOT_RE.search(t):
        return None

    t = " ".join(
        line for line in t.splitlines()
        if not line.lstrip().startswith(">")
    )

    t = re.sub(r"\s+", " ", URL_RE.sub(" ", t)).strip()
    if not any(c.isalpha() for c in t):
        return None

    return t


def prepare_comment(raw):
    t = clean_comment(raw)

    if t is None:
        return None, "bad"

    w = t.split()

    if len(w) < MIN_COMMENT_WORDS:
        return None, "short"

    # Prioritising explicit labels like ENTJ, INFP, etc
    if TYPE_RE.search(t):
        if MASK_LEAKY_COMMENTS:
            return None, "leaky"

    # Other references
    other_leak_patterns = (
        FUNC_RE,
        VOCAB_RE,
    )

    if MASK_LEAKY_COMMENTS and any(
        pattern.search(t) for pattern in other_leak_patterns
    ):
        return None, "leaky"

    # Placeholders
    if PLACEHOLDER_RE.search(t):
        return " ".join(w[:MAX_COMMENT_WORDS]), "with placeholder"

    # Ok comments
    return " ".join(w[:MAX_COMMENT_WORDS]), "ok"


Running cleaning and masking functions on the final dataset:

In [ ]:
dataset_path = "/content/drive/MyDrive/Thesis_Elena_Maggiore/Datasets_PANDORA/comments_final.csv"

def replace_leaks(text):
    for pattern, placeholder in REPLACEMENTS:
        text = pattern.sub(placeholder, text)

    text = VOCAB_RE.sub("TYPE_MENTION", text)

    return text


def clean_for_dataset(raw):
    cleaned, status = prepare_comment(raw)

    if status in ("bad", "short"):
        return None, status

    if status == "leaky":
        text = clean_comment(raw)

        if text is None:
            return None, "bad"

        text = replace_leaks(text)
        return text, "leaky"

    return cleaned, status


# Processing single dataset
print("\n" + "=" * 65)
print(f"Working on dataset: {dataset_path}")
print("=" * 65)

if not os.path.isfile(dataset_path):
    print(f"WARNING: file not found: {dataset_path}")
else:
    temp_path = dataset_path + ".cleaning_tmp"

    stats = Counter()
    total_input = 0
    total_output = 0

    try:
        header = pd.read_csv(dataset_path, nrows=0)
        columns = list(header.columns)
        del header
        gc.collect()

        comment_col = next(
            (c for c in ["body", "comment", "text"] if c in columns),
            columns[2] if len(columns) > 2 else None
        )

        if comment_col is None:
            raise ValueError(
                "Can't find comment columns."
            )

        print(f"Comment column: {comment_col}")
        print(f"Chunk size: {CHUNK_SIZE:,} rows")
        print("Working...")

        # Batch writing, to avoid RAM OOM
        first_chunk = True

        for chunk in pd.read_csv(
            dataset_path,
            chunksize=CHUNK_SIZE,
            low_memory=True
        ):
            total_input += len(chunk)

            cleaned_texts = []
            keep_positions = []

            for position, raw in enumerate(chunk[comment_col].values):
                cleaned, status = clean_for_dataset(raw)
                stats[status] += 1

                if status in ("bad", "short") or cleaned is None:
                    continue

                cleaned_texts.append(cleaned)
                keep_positions.append(position)

            cleaned_chunk = chunk.iloc[keep_positions].copy()

            cleaned_chunk[comment_col] = cleaned_texts

            total_output += len(cleaned_chunk)

            cleaned_chunk.to_csv(
                temp_path,
                mode="w" if first_chunk else "a",
                header=first_chunk,
                index=False,
                encoding="utf-8"
            )

            first_chunk = False

            del chunk, cleaned_chunk
            del cleaned_texts, keep_positions
            gc.collect()

            if torch is not None and torch.cuda.is_available():
                torch.cuda.empty_cache()

            print(
                f"Input: {total_input:,} comments | "
                f"Output: {total_output:,} comments",
                end="\r"
            )

        # In case of empty CSV
        if first_chunk:
            pd.DataFrame(columns=columns).to_csv(
                temp_path,
                index=False,
                encoding="utf-8"
            )

        os.replace(temp_path, dataset_path)

        print("\n\n Dataset successfully saved.")
        print(f"Starting rows: {total_input:,}")
        print(f"Final rows:   {total_output:,}")
        print(f"Discarded rows:  {total_input - total_output:,}")

        print("\nStats:")
        for status, count in sorted(stats.items()):
            percentage = (
                count / total_input * 100 if total_input else 0
            )
            print(
                f"  {status.upper()}: {count:,} "
                f"({percentage:.2f}%)"
            )

    except Exception as e:
        print(f"\nWARNING: {e}")
        print("Original file was not changed.")

        # Remove temporary file
        if os.path.exists(temp_path):
            try:
                os.remove(temp_path)
            except OSError:
                print(f"WARNING: error in deleting temporary file at {temp_path}")

    finally:
        gc.collect()
        if torch is not None and torch.cuda.is_available():
            torch.cuda.empty_cache()
        print("RAM freed.")

print("\n" + "=" * 65)
print("Completed")
print("=" * 65)


Working on dataset: /content/drive/MyDrive/Thesis_Elena_Maggiore/Datasets_PANDORA/comments_final.csv
Comment column: body
Chunk size: 20,000 rows
Working...
Input: 1,045,374 comments | Output: 895,742 comments

 Dataset successfully saved.
Starting rows: 1,045,374
Final rows:   895,742
Discarded rows:  149,632

Stats:
  BAD: 7,200 (0.69%)
  LEAKY: 55,686 (5.33%)
  OK: 817,458 (78.20%)
  SHORT: 142,432 (13.62%)
  WITH PLACEHOLDER: 22,598 (2.16%)
RAM freed.

Completed


Loading file *author_profiles*:

In [ ]:
author_profiles_path = '/content/drive/MyDrive/Thesis_Elena_Maggiore/Datasets_PANDORA/author_profiles.csv'

try:
    df_authors = pd.read_csv(author_profiles_path)
    print('\nauthor_profiles.csv loaded successfully to GPU.')
    display(df_authors.drop(columns="author").head(1))
except FileNotFoundError:
    print(f"Error: '{author_profiles_path}' not found. Please ensure the file is in your Google Drive and you have correctly mounted it.")
except Exception as e:
    print(f"An error occurred while loading '{author_profiles_path}': {e}.")


mbti_columns = ["mbti", "introverted", "intuitive", "thinking", "perceiving"]
big_five_columns = ["agreeableness", "openness", "conscientiousness", "extraversion", "neuroticism"]
all_columns = mbti_columns + big_five_columns

# Final dataset
final_dataset = df_authors.dropna(subset=all_columns)
print(f"Shape of 'final_dataset': {final_dataset.shape}")
print("Head of 'final_dataset':")
display(final_dataset.drop(columns="author").head(1))

#Cleaning
mbti_columns = ["mbti", "introverted", "intuitive", "thinking", "perceiving"]
big_five_columns = ["agreeableness", "openness", "conscientiousness", "extraversion", "neuroticism"]

#Counting rows: was the partitioning successful?
len_final = len(final_dataset)

cond_mbti = df_authors[mbti_columns].notna().all(axis=1)
cond_b5 = df_authors[big_five_columns].notna().all(axis=1)
len_initial_dataset = len(df_authors[cond_mbti | cond_b5])

print(f"Authors in 'final_dataset': {len_final}")
print(f"Valid authors in the starting dataset: {len_initial_dataset}")


author_profiles.csv loaded successfully to GPU.


,mbti,introverted,intuitive,thinking,perceiving,gender,age,enneagram,country,state,...,enneagram_wing,is_native_english_country,predicted_test,test_name,test_scale,16pers_ta,test_result_type,is_female,is_female_pred,is_female_proba
0,intp,1.0,1.0,1.0,1.0,m,19.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,1,0.531728


Shape of 'final_dataset': (377, 38)
Head of 'final_dataset':


,mbti,introverted,intuitive,thinking,perceiving,gender,age,enneagram,country,state,...,enneagram_wing,is_native_english_country,predicted_test,test_name,test_scale,16pers_ta,test_result_type,is_female,is_female_pred,is_female_proba
7,entp,0.0,1.0,1.0,1.0,m,21.0,3w4,NaN,NaN,...,4.0,NaN,1.0,qualtrics,percentiles,NaN,description,0.0,0,0.027725


Authors in 'final_dataset': 377
Valid authors in the starting dataset: 10258


Merging the dataset with authors' profiles:

In [ ]:
import os
import gc
import json
import sqlite3
import tempfile
import pandas as pd

# Path
base_path = (
    '/content/drive/MyDrive/Thesis_Elena_Maggiore/Datasets_PANDORA'
)

# Running on the final dataset
variable_name = "final_dataset"
comments_path = os.path.join(base_path, "comments_final.csv")
output_path = os.path.join(base_path, "final_complete_dataset.csv")
batch_size = 300
CHUNK_SIZE = 20_000


# Clearing RAM
def clear_memory():
    gc.collect()

    try:
        import torch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    except (ImportError, RuntimeError):
        pass

    try:
        import cupy
        cupy.get_default_memory_pool().free_all_blocks()
        cupy.get_default_pinned_memory_pool().free_all_blocks()
    except (ImportError, RuntimeError):
        pass


# Partitioning

print("\n" + "=" * 65)
print(f"Partitioning: {variable_name}")
print(f"Comments:   {comments_path}")
print(f"Output:     {output_path}")
print("=" * 65)

if variable_name not in globals():
    raise NameError(
        f"Variable '{variable_name}' does not exist. "
    )

if not os.path.exists(comments_path):
    raise FileNotFoundError(
        f"File not found: {comments_path}"
    )

source_dataset = globals()[variable_name]

if hasattr(source_dataset, "to_pandas"):
    authors_df = source_dataset.to_pandas()
else:
    authors_df = source_dataset.copy()

del source_dataset
clear_memory()

if "author" not in authors_df.columns:
    raise KeyError(
        f"Partition '{variable_name}' has no 'author' column."
    )

authors_df["author"] = authors_df["author"].astype("string")

author_values = authors_df["author"].dropna().unique().tolist()
author_set = set(author_values)

print(f"Authors in partition: {len(authors_df):,}")
print(f"Valid authors:  {len(author_set):,}")

fd, sqlite_path = tempfile.mkstemp(
    prefix="comments_merge_",
    suffix=".sqlite",
    dir="/content"
)
os.close(fd)

output_tmp = output_path + ".tmp"

conn = None
try:

    conn = sqlite3.connect(sqlite_path)

    conn.execute("""
        CREATE TABLE comments (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            author TEXT NOT NULL,
            body TEXT,
            subreddit TEXT
        )
    """)

    print("\nBatch importing comments:")

    imported_comments = 0

    for chunk in pd.read_csv(
        comments_path,
        chunksize=CHUNK_SIZE,
        low_memory=True,
        dtype={"author": "string"}
    ):

        required_columns = {"author", "body", "subreddit"}
        missing = required_columns - set(chunk.columns)

        if missing:
            raise KeyError(
                f"File {comments_path} has missing column: {missing}"
            )

        # Con
        mask = chunk["author"].isin(author_set)

        if mask.any():
            selected = chunk.loc[
                mask, ["author", "body", "subreddit"]
            ].copy()

            selected["body"] = (
                selected["body"].fillna("").astype(str)
            )
            selected["subreddit"] = (
                selected["subreddit"].fillna("").astype(str)
            )

            records = (
                (str(author), body, subreddit)
                for author, body, subreddit
                in selected.itertuples(index=False, name=None)
                if pd.notna(author)
            )

            conn.executemany(
                """
                INSERT INTO comments (author, body, subreddit)
                VALUES (?, ?, ?)
                """,
                records
            )

            imported_comments += len(selected)

            del selected, records

        del chunk, mask
        clear_memory()

    conn.commit()

    print(f"Imported: {imported_comments:,} comments")

    conn.execute(
        "CREATE INDEX idx_comments_author ON comments(author)"
    )
    conn.commit()


    print("\nMerge and batch save")

    if os.path.exists(output_tmp):
        os.remove(output_tmp)

    total_batches = (
        len(authors_df) + batch_size - 1
    ) // batch_size

    for batch_number, start in enumerate(
        range(0, len(authors_df), batch_size), start=1
    ):

        author_batch = authors_df.iloc[
            start:start + batch_size
        ].copy()

        batch_authors = (
            author_batch["author"].dropna().unique().tolist()
        )

        aggregated = {}

        if batch_authors:
            placeholders = ",".join(
                "?" for _ in batch_authors
            )

            query = f"""
                SELECT author, body, subreddit
                FROM comments
                WHERE author IN ({placeholders})
                ORDER BY author, id
            """

            cursor = conn.execute(query, batch_authors)

            for author, body, subreddit in cursor:
                if author not in aggregated:
                    aggregated[author] = {
                        "comments": [],
                        "subreddits": []
                    }

                aggregated[author]["comments"].append(
                    body if body is not None else ""
                )
                aggregated[author]["subreddits"].append(
                    subreddit if subreddit is not None else ""
                )

        author_batch["comments"] = author_batch["author"].map(
            lambda author: json.dumps(
                aggregated.get(
                    author,
                    {"comments": [], "subreddits": []}
                )["comments"],
                ensure_ascii=False
            )
        )

        author_batch["subreddits"] = author_batch["author"].map(
            lambda author: json.dumps(
                aggregated.get(
                    author,
                    {"comments": [], "subreddits": []}
                )["subreddits"],
                ensure_ascii=False
            )
        )

        author_batch.to_csv(
            output_tmp,
            index=False,
            mode="w" if batch_number == 1 else "a",
            header=(batch_number == 1),
            encoding="utf-8"
        )

        print(
            f"Batch {batch_number}/{total_batches} completed"
            f"({len(author_batch):,} authors)"
        )

        del author_batch, aggregated, batch_authors
        clear_memory()

    if len(authors_df) == 0:
        empty_df = authors_df.copy()
        empty_df["comments"] = pd.Series(dtype="object")
        empty_df["subreddits"] = pd.Series(dtype="object")
        empty_df.to_csv(
            output_tmp, index=False, encoding="utf-8"
        )
        del empty_df

    conn.close()
    conn = None

    os.replace(output_tmp, output_path)

    print(f"\nCompleted: {output_path}")

except Exception:
    if os.path.exists(output_tmp):
        os.remove(output_tmp)
    raise

finally:
    if conn is not None:
        conn.close()

    if os.path.exists(sqlite_path):
        os.remove(sqlite_path)

    clear_memory()


Partitioning: final_dataset
Comments:   /content/drive/MyDrive/Thesis_Elena_Maggiore/Datasets_PANDORA/comments_final.csv
Output:     /content/drive/MyDrive/Thesis_Elena_Maggiore/Datasets_PANDORA/final_complete_dataset.csv
Authors in partition: 377
Valid authors:  377

Batch importing comments:
Imported: 895,742 comments

Merge and batch save
Batch 1/2 completed(300 authors)
Batch 2/2 completed(77 authors)

Completed: /content/drive/MyDrive/Thesis_Elena_Maggiore/Datasets_PANDORA/final_complete_dataset.csv


Preparing the file for the GitHub commit:

In [ ]:
!pip -q install nbstripout

SRC = "/content/drive/MyDrive/Thesis_Elena_Maggiore/Code/PREPROCESSING: masking.ipynb"
DST = "/content/drive/MyDrive/Thesis_Elena_Maggiore/Code/GIT_READY/PREPROCESSING_masking.ipynb"

!mkdir -p "/content/drive/MyDrive/Thesis_Elena_Maggiore/Code/GIT_READY"
!cp "{SRC}" "{DST}"
!nbstripout --extra-keys "metadata.colab cell.metadata.colab cell.metadata.executionInfo cell.metadata.outputId" "{DST}"
!grep -c "displayName\|userId" "{DST}"
!grep -c '"outputs": \[\]' "{DST}"

1
7
